# Estimating Power Plant Capacity Factors from Plant Characteristics
**A CRISP-DM analysis of the WRI Global Power Plant Database (v1.3.0)**

Author: Philip Dawson-Gariepy · Module: Data Analytics (CA2) · Munster Technoligical University

## 0. Setup
This section imports the libraries used throughout the notebook, records software versions for repoducibility, and fixes a random seed so results can be repeated.

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

RANDOM_STATE = 42 # fixed seed used for every split and model
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 50)

print("Python       ", sys.version.split()[0])
print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("matplotlib  ", matplotlib.__version__)
print("seaborn     ", sns.__version__)
print("scikit-learn", sklearn.__version__)

Python        3.14.7
pandas       3.0.6
numpy        2.5.4
matplotlib   3.11.2
seaborn      0.13.2
scikit-learn 1.9.1


### Load the raw data
This raw CSV is not stored in the repository. See `data_instructions/README.md` for how to download it and where to place it. The path below is relative to the `notebooks/` folder. The shape check confirms the expected v1.3.0 file was loaded.

In [2]:
DATA_PATH = "../data/global_power_plant_database.csv"

raw = pd.read_csv(DATA_PATH, low_memory=False)
assert raw.shape == (34936, 36), f"Unexpected shape: {raw.shape} - check data_instrcutions/README.md"

df = raw.copy() # work on a copy; `raw` is never modified
df.head()

,country,country_long,name,gppd_idnr,capacity_mw,latitude,longitude,primary_fuel,other_fuel1,other_fuel2,other_fuel3,commissioning_year,owner,source,url,geolocation_source,wepp_id,year_of_capacity_data,generation_gwh_2013,generation_gwh_2014,generation_gwh_2015,generation_gwh_2016,generation_gwh_2017,generation_gwh_2018,generation_gwh_2019,generation_data_source,estimated_generation_gwh_2013,estimated_generation_gwh_2014,estimated_generation_gwh_2015,estimated_generation_gwh_2016,estimated_generation_gwh_2017,estimated_generation_note_2013,estimated_generation_note_2014,estimated_generation_note_2015,estimated_generation_note_2016,estimated_generation_note_2017
0,AFG,Afghanistan,Kajaki Hydroelectric Power Plant Afghanistan,GEODB0040538,33.0,32.322,65.1190,Hydro,NaN,NaN,NaN,NaN,NaN,GEODB,http://globalenergyobservatory.org,GEODB,1009793,2017.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,123.77,162.90,97.39,137.76,119.50,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1
1,AFG,Afghanistan,Kandahar DOG,WKS0070144,10.0,31.670,65.7950,Solar,NaN,NaN,NaN,NaN,NaN,Wiki-Solar,https://www.wiki-solar.org,Wiki-Solar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.43,17.48,18.25,17.70,18.29,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE
2,AFG,Afghanistan,Kandahar JOL,WKS0071196,10.0,31.623,65.7920,Solar,NaN,NaN,NaN,NaN,NaN,Wiki-Solar,https://www.wiki-solar.org,Wiki-Solar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.64,17.58,19.10,17.62,18.72,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE
3,AFG,Afghanistan,Mahipar Hydroelectric Power Plant Afghanistan,GEODB0040541,66.0,34.556,69.4787,Hydro,NaN,NaN,NaN,NaN,NaN,GEODB,http://globalenergyobservatory.org,GEODB,1009795,2017.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,225.06,203.55,146.90,230.18,174.91,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1
4,AFG,Afghanistan,Naghlu Dam Hydroelectric Power Plant Afghanistan,GEODB0040534,100.0,34.641,69.7170,Hydro,NaN,NaN,NaN,NaN,NaN,GEODB,http://globalenergyobservatory.org,GEODB,1009797,2017.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,406.16,357.22,270.99,395.38,350.80,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1


---
# Phase 1: Business Understanding

## 1.0 Background: how power plants work (for non-specialists)
This project uses some electrical engineering ideas. This section explains them in everyday terms, so the rest of the notebook can be followed without an engineering background.

### Power versus energy
A kettle might be rated at **2 kilowatts (kW)**. That is its *power*: how fast it can use electricity. If it runs for half an hour, it uses **1 kilowatt-hour (kWh)** of *energy*. Power plants work the same way, on a much larger scale:

| Term | Meaning | Kettle example | Power plant example |
| --- | --- | --- | --- |
| **Capacity** (MW, megawatts) | The most power the plant can produce at any moment: its "rating" | 2 kW | A 100 MW wind farm |
| **Generation** (GWh, gigawatt-hours) | The energy the plant actually produced over a period | 1 kWh in half an hour | 219 GWh in a year |

1 MW = 1,000 kW, and 1 GWh = 1,000,000 kWh. A 100 MW plant running flat out for one hour produces 100 MWh (0.1 GWh).

### Capacity factor: the key idea
No plant runs at full power all year. The **capacity factor** measures how hard a plant actually worked, compared with the most it could have produced:

```
capacity factor = energy actually produced ÷ energy it could have produced at full power all year
```

A year has 8,760 hours, so a 100 MW plant could produce at most 100 × 8,760 = 876,000 MWh, or **876 GWh**. If it actually produced 219 GWh, its capacity factor is 219 ÷ 876 = **0.25**: on average it ran at a quarter of its maximum. A capacity factor always lies between 0 (never ran) and 1 (ran at full power every hour of the year). In practice even the most heavily used plants stay below about 0.9, because they must stop for maintenance [1].

### Why different plants run so differently
How hard a plant runs depends mostly on what kind of plant it is. The US Energy Information Administration describes the same broad patterns [2]:

| Plant type | How it behaves | What to expect |
| --- | --- | --- |
| **Nuclear** | Runs almost non-stop. Fuel is cheap but stopping and restarting is slow and costly, so it stays on except for refuelling and maintenance. | Very high |
| **Coal** | Built to run for long periods, but used less when cheaper electricity is available. | Medium to high |
| **Gas** | Two very different kinds share this label. *Combined-cycle* plants are efficient and run a lot; *gas turbines* ("peakers") start quickly and run only when demand is highest. The dataset does not say which kind a plant is. | Anywhere from very low to high, so hard to predict |
| **Oil** | Mostly backup or emergency plants, used rarely. | Very low |
| **Hydro** | Depends on rain, river flow and snowmelt, which vary by region and year. | Medium |
| **Wind** | Only produces when the wind blows; cannot be switched on at will. | Low to medium |
| **Solar** | Only produces in daylight; output depends on how far from the equator the plant is and how cloudy it is. | Low |
| **Geothermal, biomass, waste** | Fuel or heat is available continuously, so these run steadily. | Medium to high |
| **Storage** (batteries, pumped hydro) | Stores electricity and releases it later, losing some along the way, so over a year it uses more than it produces. | Not meaningful, so excluded |

### The grid decides who runs
Demand for electricity rises and falls through the day. Grid operators generally call on the cheapest available plants first and bring in more expensive ones only when demand is high (often called the *merit order*). This means **how hard a plant runs depends partly on the system around it**: fuel prices, other plants on the same grid, and national policy. None of that information is in the dataset, which limits how accurate any estimate can be.

### Why some values are negative
Plants use some of their own electricity to run pumps, fans, lighting and controls. Published figures are usually *net* generation: output minus the plant's own use. A backup plant that barely runs during a year can therefore report **negative** net generation. This explains some of the negative values found in the data.

### What this means for the model
The engineering background suggests which plant characteristics should carry useful information:
- **Fuel type**: the strongest expected signal, for the reasons in the table above.
- **Latitude**: plants closer to the equator receive more sunlight, which matters most for solar.
- **Commissioning year (age)**: older plants are often less efficient and may be called on less; newer wind and solar technology tends to perform better.
- **Capacity**: plant size can hint at its role in the grid, for example small backup units versus large baseload stations.
- **Country**: a rough stand-in for climate, the national grid mix and energy policy.

## 1.1 Context and problem
To understand a power plant's impact on the environment, we need to know how much electricity it actually produced, not just how big it is. Two plants of the same size can have very different emissions if one runs all year and the other only on the coldest winter evenings.

This information is surprisingly hard to find. Most countries do not publish how much electricity each individual plant produces, including many developing countries where electricity production is expected to grow fastest [1]. The dataset used here, the Global Power Plant Database [3], shows the gap clearly:

- It lists **34,936 power plants** in 167 countries.
- Only **9,500** of them (27%) report their 2017 generation, and **8,370** of those are in the United States.
- The plants that do **not** report make up about **two-thirds of the world's generating capacity** in the database.

**The problem:** for most of the world's power plants, we know their size, fuel and location, but not how much electricity they produced. Anyone estimating national electricity output or emissions has to fill that gap with estimates, and it is unclear how trustworthy those estimates are.

## 1.2 Stakeholders and users
| Role | Who | What they care about |
| --- | --- | --- |
| Data suppliers | The World Resources Institute (WRI) and partners, who gathered the data from national sources such as the US Energy Information Administration, India's Central Electricity Authority and Australia's Clean Energy Regulator | That the data is accurate and openly reused |
| **Primary user** | **Analysts who estimate how much electricity, and therefore how much pollution, power plants produce in countries that do not publish this** (for example climate research groups and organisations tracking emissions from the energy sector) | Estimates they can rely on, and a clear warning where they cannot |
| Secondary users | Energy planners and researchers comparing countries' power systems | Figures that are comparable between countries |
| Affected parties | Countries and plant owners whose output or emissions may be over- or under-estimated | Being represented fairly; wrong estimates could distort policy debates or damage reputations |

Throughout the project, the focus is on the primary user: someone with a list of plants whose size, fuel and location are known but whose actual output is not, who needs a sensible estimate for each one.

## 1.3 Practical objective
The aim is to help analysts **fill in missing power plant output figures with estimates whose accuracy is known**. That means the analyst can produce output and emissions figures for plants that do not report them, and can tell which estimates are reliable enough to use and which should be treated with caution.

The result supports human analysis and reporting. It is not meant to make automatic decisions.

## 1.4 Analytics objective
Build a model that estimates each plant's **2017 capacity factor** from information known for most plants (fuel type, size, age and location), and measure how accurate it is.

- **Unit of analysis:** one power plant (one row of the dataset).
- **What is predicted (the target):** the plant's 2017 capacity factor, calculated as

  `capacity_factor = generation_gwh_2017 / (capacity_mw × 8.76)`

  The 8.76 converts capacity in MW into the maximum energy in GWh the plant could produce in a year (8,760 hours ÷ 1,000). See section 1.0 for a worked example.
- **Type of task:** regression, because the target is a number rather than a category.
- **Why capacity factor rather than raw generation:** generation mostly reflects plant size. A model predicting generation directly would mainly learn that big plants produce more, and its errors would be dominated by the few largest plants. Capacity factor removes the effect of size, so a 2 MW plant and a 2,000 MW plant can be compared fairly. WRI made the same choice [1]. Once a capacity factor is estimated, it converts back to generation by multiplying by capacity × 8.76.
- **Why 2017:** it is the most recent year with both reported generation and WRI's own published estimates, which are used as a point of comparison.

### Three ways of using the data
Following the lecturer's suggestion, the data is used in three ways, each answering a different question:

| Scenario | Plants used | Question it answers | How it is checked |
| --- | --- | --- | --- |
| **A. Known plants: testing the method** | The ~9,400 plants that report 2017 generation | Can the model reproduce real values it has not seen? | The plants are split into a training set and a test set. The model learns from the training set and is scored on the test set, where the true answer is known. **This is the only scenario where accuracy can be measured directly.** |
| **B. Unknown plants: filling the gap** | The ~25,400 plants with no reported generation | What does the model estimate for the plants that actually need it? | There is no true answer to check against. Instead: are the estimates physically sensible (section 1.0)? How closely do they agree with WRI's estimates? And how different are these plants from the ones the model learned from? |
| **C. The whole fleet: the complete picture** | All ~34,800 plants (excluding storage) | What does total electricity output look like when real figures and estimates are combined? | Totals by country and fuel, showing what share of each country's total comes from real reports versus estimates, so users can judge how far to trust national totals. |

## 1.5 Success criteria
**Technical criteria**

| ID | Scenario | Criterion | How it is measured |
| --- | --- | --- | --- |
| T1 | A | The model beats a simple rule of thumb: "predict the typical (median) capacity factor for the plant's fuel type", learned from the training data | Lower mean absolute error (MAE) on the test set |
| T2 | A | The model is compared fairly with WRI's published 2017 estimates for the same test plants | MAE for each fuel type, model vs. WRI |
| T3 | A | Errors are reported in units people understand | MAE in capacity factor and in GWh |
| T4 | A | Accuracy is checked separately for US and non-US plants | MAE for each group |
| T5 | B | Estimates are physically plausible | All estimates between 0 and 1; typical values per fuel type consistent with section 1.0 and with the reporting plants |
| T6 | B | Agreement with WRI's estimates is measured, keeping in mind that agreement is not the same as accuracy | Average difference between model and WRI estimates, by fuel type |
| T7 | C | Combined totals are transparent | For each country, the share of total estimated output that comes from real reports |

*Mean absolute error (MAE)* is the average size of the error, ignoring whether the estimate was too high or too low. An MAE of 0.10 means estimates are off by 0.10 capacity factor on average.

**Practical criteria**

| ID | Criterion |
| --- | --- |
| P1 | In scenario A, an average error (MAE) of **0.10 or less**. For a typical plant of about 18 MW, this means being off by roughly 15 GWh a year; for a 100 MW plant, about 88 GWh. |
| P2 | The main factors behind each estimate can be explained in plain language to a non-specialist. |
| P3 | The fuel types and regions where estimates are unreliable are clearly identified, so a user knows where not to rely on them. |

**Early feasibility check.** For the 9,275 reporting plants with a valid capacity factor (storage excluded, impossible values removed), the simple fuel-type rule of thumb is off by about 0.14 on average, while WRI's published estimates are off by about 0.17. WRI does better for solar, wind, hydro and nuclear, but much worse for gas and waste, which matches the engineering expectation that gas is the hardest type to predict (section 1.0). The rule-of-thumb figure is optimistic because it was calculated on the same plants it is scored on; it is recalculated properly in Phase 4. Meeting P1 needs roughly a 30% improvement on the rule of thumb. P1 may be revised after Data Understanding, and any revision will be recorded.

The cell below reproduces the feasibility figures quoted above.

In [3]:
# Early feasibility check (in-sample; recalculated properly in Phase 4)
chk = df[df["generation_gwh_2017"].notna() & (df["primary_fuel"] != "Storage")].copy()
chk["cf"] = chk["generation_gwh_2017"] / (chk["capacity_mw"] * 8.76)
chk = chk[chk["cf"].between(0, 1)]
chk["wri_cf"] = chk["estimated_generation_gwh_2017"] / (chk["capacity_mw"] * 8.76)

baseline = chk.groupby("primary_fuel")["cf"].transform("median")
has_wri = chk["wri_cf"].notna()

print(f"Plants with valid 2017 capacity factor: {len(chk):,}")
print(f"Fuel-median baseline MAE: {(chk['cf'] - baseline).abs().mean():.3f}")
print(f"WRI estimate MAE:         {(chk.loc[has_wri, 'cf'] - chk.loc[has_wri, 'wri_cf']).abs().mean():.3f}")

Plants with valid 2017 capacity factor: 9,275
Fuel-median baseline MAE: 0.141
WRI estimate MAE:         0.171


## 1.6 Constraints and assumptions
**Constraints**
- **Old data:** the database's final version is v1.3.0 (June 2021) and is no longer updated, so findings describe power plants around 2017.
- **Uneven coverage:** about 88% of the plants that report generation are in the US, while 94% of the plants that do not report are outside it. The model will mostly learn from US plants but is needed mostly elsewhere. Scenario B is designed to expose this.
- **Missing age information:** commissioning year is missing for about two-thirds of the non-reporting plants, compared with about 5% of reporting plants. Any feature based on age will need careful handling in scenario B.
- **Missing causes:** many things that decide how hard a plant runs, such as fuel prices, grid decisions, plant efficiency and the exact technology (for example, which kind of gas plant), are not in the data (section 1.0; [1]).
- **Time:** an eight-week project, which favours a simple, well-tested model over a complicated one.
- **Privacy:** the data contains no personal information. The `owner` column names companies and is not used.

**Assumptions**
- Plant characteristics contain enough information to estimate capacity factor better than fuel type alone.
- Reported generation is broadly accurate once physically impossible values are removed; the original sources cannot be independently checked.
- Storage plants are out of scope, because they use more electricity than they produce by design (section 1.0).

## 1.7 Initial project plan
| Week | CRISP-DM phase | Main activities |
| --- | --- | --- |
| 6 | Proposal | Agree the problem, data, unit of analysis, method and risks with the lecturer |
| 7 | Business Understanding | Background, objectives, success criteria, plan; repository and README |
| 8 | Data Understanding | Data collection, description, exploration and quality reports; data dictionary |
| 9 | Data Preparation | Drop unneeded columns, build the target, clean values, split into scenarios A/B/C, train/test split, preparation log |
| 10 | Modelling | Scenario A: rule-of-thumb baseline, linear regression, decision tree |
| 11 | Evaluation | Scenario A accuracy; scenario B plausibility and agreement with WRI; scenario C totals; paper draft |
| 12 | Deployment and submission | Recommendations, reflection, AI statement, final paper |

**Proposed technique:** regression. It starts from the fuel-type rule of thumb, then tries multiple linear regression (easy to explain), then a decision tree (can capture the different behaviour of each fuel type, as described in section 1.0). The final model is chosen on its scenario A test results.

---
# Phase 2: Data Understanding

## 2.1 Initial Data Collection Report
*Source, publisher, access date, licence, how the data was compiled, population, time period, geography, known exclusions, files acquired.*

## 2.2 Data Description Report
**Size and unit of analysis:** 34,936 rows and 36 columns. Each row is one power plant, identified by a unique code (`gppd_idnr`).

### Column review
Not all 36 columns are useful for this project. Many repeat the same information, describe where the record came from rather than the plant itself, or cover years other than the 2017 target. Each column was reviewed and assigned a role. The checks in the code cell below back up the reasons given.

| Group | Columns | Decision | Reason |
| --- | --- | --- | --- |
| Identifier | `gppd_idnr` | **Keep** (not a model input) | Unique code for each plant, used to look a plant up in the raw data |
| Plant characteristics | `country`, `capacity_mw`, `latitude`, `longitude`, `primary_fuel`, `commissioning_year` | **Keep** (model inputs) | Physical facts about each plant, known for most plants (section 1.0 explains why each may matter) |
| Secondary fuel | `other_fuel1` | **Keep**, converted to a yes/no flag in Phase 3 | Shows whether a plant can burn a second fuel |
| Target | `generation_gwh_2017` | **Keep** | Used to calculate the 2017 capacity factor |
| Benchmark | `estimated_generation_gwh_2017`, `estimated_generation_note_2017` | **Keep** (never model inputs) | WRI's own 2017 estimate and the method used to make it, used only for comparison |
| Duplicate | `country_long` | Drop | The full country name: a one-to-one copy of the `country` code |
| Labels and links | `name`, `url`, `wepp_id` | Drop | A text label, a web link and a reference number for another database. None describe the plant's behaviour |
| Record-keeping | `source`, `geolocation_source`, `year_of_capacity_data` | Drop | Where and when the record was collected, not facts about the plant. `source` would also act as a hidden clue to whether a plant reports generation |
| Mostly empty | `other_fuel2`, `other_fuel3`, `owner` | Drop | 99% and over 99% empty; `owner` is 40% empty with over 10,000 different company names |
| Other years | `generation_gwh_2013` to `2016`, `2018`, `2019` | Drop | Real output for years other than 2017. Only about 7% of plants without 2017 figures (scenario B) have any of these years, so they cannot serve as inputs, and using them would only work for plants that already report |
| Data source of output | `generation_data_source` | Drop | Only filled in when a plant reports generation, so it would give away which plants are in scenario A |
| Older WRI estimates | `estimated_generation_gwh_2013` to `2016`, `estimated_generation_note_2013` to `2016` | Drop | Earlier-year versions of WRI's estimates, almost identical to the 2017 estimate (correlation 0.98 or higher) |

**Result:** 11 columns kept, 25 dropped. The raw data (`raw`) is never changed; the reduced table is created in Phase 3.

### Data dictionary (columns kept)
| Column | Meaning | Type | Units | Valid range | Role |
| --- | --- | --- | --- | --- | --- |
| `gppd_idnr` | Unique plant identifier | Text | - | Unique | Identifier |
| `country` | Country code (ISO 3166-1 alpha-3) | Category | - | 167 codes | Input |
| `capacity_mw` | Maximum power output (see 1.0) | Number | MW | Above 0 | Input; used in target |
| `latitude` | Distance north (+) or south (-) of the equator | Number | Degrees | -90 to 90 | Input |
| `longitude` | Distance east (+) or west (-) of Greenwich | Number | Degrees | -180 to 180 | Input |
| `primary_fuel` | Main energy source | Category | - | 15 types | Input |
| `other_fuel1` | Second fuel, if any | Category | - | Mostly empty | Converted to a yes/no flag |
| `commissioning_year` | Year the plant started operating (may be an average for multi-unit plants) | Number | Year | About 1890 to 2020 | Input |
| `generation_gwh_2017` | Electricity produced in 2017, net of the plant's own use | Number | GWh | Can be slightly negative (see 1.0) | Target source |
| `estimated_generation_gwh_2017` | WRI's estimated 2017 generation | Number | GWh | 0 or above | Benchmark only |
| `estimated_generation_note_2017` | Method WRI used for that estimate | Category | - | 6 labels | Benchmark only |

*Valid ranges are based on the dataset documentation [3] and on physical limits; actual values are checked in the Data Quality Report (2.4).*

The checks below support the reasons given in the column review.

In [4]:
# Evidence for the column review
# 1. country and country_long are a one-to-one match
one_to_one = (df.groupby("country")["country_long"].nunique().max() == 1
              and df.groupby("country_long")["country"].nunique().max() == 1)
print("country <-> country_long one-to-one:", one_to_one)

# 2. gppd_idnr is a unique identifier
print("gppd_idnr unique:", df["gppd_idnr"].is_unique)

# 3. How empty are the columns being dropped for missingness?
print("\nShare missing:")
print((df[["other_fuel2", "other_fuel3", "owner"]].isna().mean() * 100).round(1).astype(str) + "%")
print("Distinct owners:", df["owner"].nunique())

# 4. generation_data_source is only filled in when generation is reported
gen_years = [f"generation_gwh_{y}" for y in range(2013, 2020)]
any_gen = df[gen_years].notna().any(axis=1)
print("\ngeneration_data_source filled, among plants with NO reported generation:",
      df.loc[~any_gen, "generation_data_source"].notna().sum())

# 5. Plants without 2017 generation rarely have other years either
no_2017 = df["generation_gwh_2017"].isna()
print("Plants without 2017 generation that have any other year:",
      df.loc[no_2017, gen_years].notna().any(axis=1).sum(), "of", no_2017.sum())

# 6. Older WRI estimates closely follow the 2017 estimate
print("\nCorrelation of older WRI estimates with the 2017 estimate:")
for y in range(2013, 2017):
    print(f"  {y}: {df[f'estimated_generation_gwh_{y}'].corr(df['estimated_generation_gwh_2017']):.3f}")

country <-> country_long one-to-one: True
gppd_idnr unique: True

Share missing:
other_fuel2    99.2%
other_fuel3    99.7%
owner          40.3%
dtype: str
Distinct owners: 10144

generation_data_source filled, among plants with NO reported generation: 0
Plants without 2017 generation that have any other year: 1900 of 25436

Correlation of older WRI estimates with the 2017 estimate:
  2013: 0.981
  2014: 0.989
  2015: 0.990
  2016: 0.994


## 2.3 Data Exploration Report
*Frequencies, descriptive statistics and charts linked to the analytical question. Interpret each output straight after it appears.*

## 2.4 Data Quality Report
*Missing values (count, %, pattern), duplicates, invalid values, outliers, representation and bias. End with a prioritised list of issues for Phase 3.*

---
# Phase 3: Data Preparation
*Only changes justified by the Data Understanding findings are made. The raw data (`raw`) is never changed, and the effect of each change is recorded below.*

## 3.1 Preparation log
| Issue | Action | Rationale / evidence | Assumption | Number affected | Risk / verification |
| --- | --- | --- | --- | --- | --- |
| 25 columns not needed (duplicates, record-keeping, other years, would reveal the answer) | Dropped | Column review and checks in 2.2 | Dropped columns hold no plant information needed by any scenario | 25 of 36 columns; 0 rows | Raw data kept unchanged; dropped columns can be restored from `raw` |
| Storage plants | Excluded from all scenarios | They use more electricity than they produce by design (1.0) | Capacity factor is not meaningful for storage | 135 plants | Count recorded in 3.2 |
| | | | | | |

## 3.2 Selection
**Columns:** keep the 11 columns chosen in 2.2.

**Rows:** remove storage plants (section 1.0), then divide the remaining plants into the three scenarios from 1.4:
- **Scenario A** (known plants): 2017 generation is reported.
- **Scenario B** (unknown plants): 2017 generation is missing.
- **Scenario C** (whole fleet): A and B together. It is assembled after modelling, when B has estimates.

Impossible capacity factors are removed from scenario A in 3.3 (Cleaning), not here.

In [5]:
KEEP = [
    "gppd_idnr",                                   # identifier
    "country", "capacity_mw", "latitude", "longitude",
    "primary_fuel", "other_fuel1", "commissioning_year",   # plant characteristics
    "generation_gwh_2017",                         # target source
    "estimated_generation_gwh_2017", "estimated_generation_note_2017",  # benchmark only
]
sel = df[KEEP].copy()
print(f"Columns: {df.shape[1]} -> {sel.shape[1]}")

# Remove storage plants
is_storage = sel["primary_fuel"] == "Storage"
print(f"Storage plants removed: {is_storage.sum()}")
sel = sel[~is_storage]

# Split into scenarios
reported = sel["generation_gwh_2017"].notna()
scen_A = sel[reported].copy()    # known plants
scen_B = sel[~reported].copy()   # unknown plants

summary = pd.DataFrame({
    "plants": [len(scen_A), len(scen_B), len(sel)],
    "capacity_GW": [scen_A["capacity_mw"].sum() / 1000, scen_B["capacity_mw"].sum() / 1000, sel["capacity_mw"].sum() / 1000],
    "share_in_US_%": [(scen_A["country"] == "USA").mean() * 100, (scen_B["country"] == "USA").mean() * 100, (sel["country"] == "USA").mean() * 100],
    "age_missing_%": [scen_A["commissioning_year"].isna().mean() * 100, scen_B["commissioning_year"].isna().mean() * 100, sel["commissioning_year"].isna().mean() * 100],
}, index=["A: known", "B: unknown", "C: whole fleet"]).round(1)
summary

Columns: 36 -> 11
Storage plants removed: 135


,plants,capacity_GW,share_in_US_%,age_missing_%
A: known,9444,1817.9,88.0,4.7
B: unknown,25357,3887.4,5.6,67.1
C: whole fleet,34801,5705.3,28.0,50.2


Scenario B has more than twice the capacity of scenario A, is almost entirely outside the US, and is missing age information for most plants. A model trained on scenario A is therefore being asked to work on plants quite different from the ones it learned from. This is examined in Phase 5.

## 3.3 Cleaning

## 3.4 Type and format correction

## 3.5 Feature construction

## 3.6 Train/test split, encoding and scaling

## 3.7 Validation

---
# Phase 4: Modelling

## 4.1 Task, inputs and target

## 4.2 Baseline

## 4.3 Primary model

## 4.4 Alternative model or setting

## 4.5 Interpretability and limitations

---
# Phase 5: Evaluation

## 5.1 Results against technical success criteria

## 5.2 Practical usefulness

## 5.3 Generalisability, error analysis and failure cases

## 5.4 Fairness, transparency and misuse

## 5.5 Decision

---
# Phase 6: Deployment and Recommendations

---
# Reflection
*What changed from the original plan? Which decision needed the most judgement? What failed and what was learned? What would you do with more time or better data? Which parts can you explain confidently without assistance?*

# Responsible AI Collaboration Statement
*300–500 words. Must address all of the following:*
1. *Tools and versions used.*
2. *Tasks for which AI assistance was used.*
3. *One or more representative prompts, or a link to a prompt log.*
4. *How code, factual claims, calculations and references were verified.*
5. *Examples of outputs that were rejected, corrected or substantially modified.*
6. *What remained your own judgement and decision.*
7. *What you learned about the strengths and limitations of AI collaboration.*

# References
[1] L. Yin, L. Byers, L. Malaguzzi Valeri, and J. Friedrich, "Estimating power plant generation in the Global Power Plant Database," World Resources Institute, Washington, DC, USA, Tech. Note, Mar. 2020. [Online]. Available: https://www.wri.org/research/estimating-power-plant-generation-global-power-plant-database (accessed Nov. 5, 2026).

[2] U.S. Energy Information Administration, "Monthly generator capacity factor data now available by fuel and technology," *Today in Energy*, Jan. 15, 2014. [Online]. Available: https://www.eia.gov/todayinenergy/detail.php?id=14611 (accessed Nov. 5, 2026).

[3] Global Energy Observatory, Google, KTH Royal Institute of Technology in Stockholm, Enipedia, and World Resources Institute, *Global Power Plant Database*, version 1.3.0, World Resources Institute, Jun. 2021. [Dataset]. Available: https://datasets.wri.org/dataset/globalpowerplantdatabase (accessed Oct. 9, 2026).